# 4. Modelo SARIMAX
El modelo SARIMAX (Seasonal ARIMA with eXogenous regressors) es una extensión del modelo SARIMA que incorpora, además de los componentes autorregresivos, de media móvil y estacionales, la posibilidad de incluir variables exógenas o predictoras externas. Se expresa como:

SARIMAX(p,d,q)(P,D,Q,s)

donde:

(p,d,q) representan los componentes no estacionales y (P,D,Q) representan los componentes estacionales.

s indica el periodo de la estacionalidad (por ejemplo, 24 para datos horarios diarios).

y adicionalmente, X representa el conjunto de variables exógenas que se incorporan al modelo.

La principal ventaja de SARIMAX frente a SARIMA es su capacidad para mejorar la predicción al incorporar información adicional relevante, como pueden ser variables temporales (día de la semana, mes...) u otras variables explicativas que influyen en la serie objetivo.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.metrics import mean_squared_error

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox


# =======================================================================================
# 0. LECTURA DEL SUPERDATASET (de precios horarios de la electricidad, generacion y climatología)
# =======================================================================================
df_dummy = pd.read_csv('../../../Dataset_Unificado/Dataset_Unificado1.csv', sep=';')
df_dummy['datetime'] = pd.to_datetime(df_dummy['datetime'])
df_dummy.set_index('datetime', inplace=True) 
# A partir de aquí, 'datetime' ya no es una columna

# =======================================================================================
# 1. PREPARACIÓN Y DIVISIÓN TEMPORAL
# =======================================================================================
X = df_dummy.drop(columns=['price'])
y = df_dummy['price']

# División cronológica estricta: entrenar con 2020-2023, validar con 2024 completo
X_train, X_test = X[X['year'] < 2024], X[X['year'] == 2024]
y_train, y_test = y[X['year'] < 2024], y[X['year'] == 2024]

# Eliminamos 'year' de los predictores porque los árboles no extrapolan bien hacia el futuro
X_train = X_train.drop(columns=['year'])
X_test = X_test.drop(columns=['year'])

# Variables categóricas temporales identificadas
cat_features = ['hour', 'month', 'dayofweek', 'is_weekend']
for col in cat_features:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')


In [ ]:
# ==============================
# 5. Elegir los órdenes p y q del modelo SARIMA
# ==============================
# En Box-Jenkins a partir de los gráficos de ACF y PACF, elegimos p y q
# En nuestro caso fueron p=2 y q=1
# En ARIMA, el orden es (p, d, q).
# El valor d e sel numero de diferenciaciones para hacer la serie estacionaria,
#  pero como la serie ya es estacionaria, no es necesario diferenciar (d=0)
#  aunque se puede poner d=1 sin que cambie el resultado matemático porque
#  ARIMA con d=1 en una serie ya estacionaria es equivalente a ARMA con d=0
p, d, q = 2, 1, 1           # parte no estacional (igual que ARIMA/ARMA)
P, D, Q, s = 2, 1, 0, 24    # parte estacional 


modelo_sarimax = SARIMAX(
    y_train,
    exog=X_train,
    order=(p, d, q),
    seasonal_order=(P, D, Q, s),
    enforce_stationarity=False,
    enforce_invertibility=False
)


# Ajustar el modelo
modelo_ajustado = modelo_sarimax.fit(low_memory=True)  # Reduce uso de memoria

# Mostrar un resumen del modelo
print(modelo_ajustado.summary())


# ==============================
# 5.2. Verificar residuos (validación)
# ==============================

# Verificar residuos (validación)
residuos = modelo_ajustado.resid

# Gráfico ACF de residuos
fig, axes = plt.subplots(2, 1, figsize=(10, 8))
plot_acf(residuos, ax=axes[0], lags=96)
axes[0].set_title('ACF de residuos del modelo SARIMA')

plot_pacf(residuos, ax=axes[1], lags=96)
axes[1].set_title('PACF de residuos del modelo SARIMA')
plt.savefig('sarima_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

# Prueba de Ljung-Box (si p-value > 0.05 los residuos son ruido blanco)
lb_test = acorr_ljungbox(residuos, lags=[10], return_df=True)
print(f"\nTEST LJUNG-BOX")
print(lb_test.to_string())
if (lb_test['lb_pvalue'] > 0.05).all():
    print("los residuos son ruido blanco")
else:
    print("los residuos NO son ruido blanco (tienen estructura/autocorrelacion)")

# ==============================
# 6. Predecir los valores del conjunto de prueba
# ==============================
# forecast recibe el número de pasos a predecir y las variables exógenas del conjunto de prueba
predicciones = modelo_ajustado.forecast(steps=len(y_test), exog=X_test)
predicciones.index = y_test.index   # para que luego se pueda gaficar bien

# ==============================
# 7. AMPLIADA - Métricas de precisión
# ==============================
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

mae  = mean_absolute_error(y_test, predicciones)
rmse = np.sqrt(mean_squared_error(y_test, predicciones))
mape = mean_absolute_percentage_error(y_test, predicciones) * 100

# Métrica adicional: R² (qué % de varianza explica el modelo)
ss_res = np.sum((y_test - predicciones) ** 2)
ss_tot = np.sum((y_test - y_test.mean()) ** 2)
r2 = 1 - (ss_res / ss_tot)

# Naive benchmark: predecir siempre el valor anterior (lag-1)
naive_pred = y_test.shift(1).dropna()
rmse_naive = np.sqrt(mean_squared_error(y_test[1:], naive_pred))

print("\n========== MÉTRICAS DE PRECISIÓN ==========")
print(f"  MAE   (Error Absoluto Medio):        {mae:.4f} €/MWh")
print(f"  RMSE  (Raíz Error Cuadrático Medio): {rmse:.4f} €/MWh")
print(f"  MAPE  (Error Porcentual Medio):       {mape:.2f}%")
print(f"  R²    (Coeficiente determinación):    {r2:.4f}")
print(f"  RMSE modelo naive (benchmark):        {rmse_naive:.4f} €/MWh")
print(f"  El modelo es {'MEJOR' if rmse < rmse_naive else 'PEOR'} que el naive")
print("============================================\n")


# ==============================
# 8. CORREGIDA - Gráficas con zoom y separadas
# ==============================
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# --- Gráfica 1: visión global (últimas 4 semanas de train + todo el test) ---
CONTEXTO = 672  # horas de entrenamiento que se muestran como contexto

ax1 = axes[0]
ax1.plot(y_train[-CONTEXTO:].index, y_train[-CONTEXTO:],
         label='Entrenamiento (contexto)', color='steelblue', alpha=0.7)
ax1.plot(y_test.index, y_test,
         label='Real (prueba)', color='blue', linewidth=1.5)
ax1.plot(y_test.index, predicciones,
         label='Predicciones SARIMA', color='red', linestyle='--', linewidth=1.5)
ax1.axvline(x=y_test.index[0], color='gray', linestyle=':', linewidth=1.5,
            label='Inicio prueba')
ax1.set_title('Vista general: últimos meses de train + período de prueba')
ax1.set_xlabel('Fecha')
ax1.set_ylabel('Precio (€/MWh)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Gráfica 2: zoom solo en el período de prueba ---
ax2 = axes[1]
ax2.plot(y_test.index, y_test,
         label='Real', color='blue', linewidth=1.5)
ax2.plot(y_test.index, predicciones,
         label='Predicciones SARIMA', color='red', linestyle='--', linewidth=1.5)
ax2.fill_between(y_test.index,
                 predicciones - rmse,
                 predicciones + rmse,
                 alpha=0.15, color='red', label=f'±1 RMSE ({rmse:.2f})')
ax2.set_title(f'Zoom período de prueba — MAPE: {mape:.2f}% | RMSE: {rmse:.2f} | R²: {r2:.3f}')
ax2.set_xlabel('Fecha')
ax2.set_ylabel('Precio (€/MWh)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sarima_prediccion.png', dpi=150, bbox_inches='tight')
plt.show()

c:\Users\Jaime_Sanchez\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
c:\Users\Jaime_Sanchez\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
c:\Users\Jaime_Sanchez\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
